In [4]:
import jax
import jax.numpy as jnp

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns  

import liesel.model as lsl
import liesel.goose as gs

from tensorflow_probability.substrates.jax.experimental import distributions as tfde
from tensorflow_probability.python.internal.backend.jax.compat import v2 as tf
import tensorflow_probability.substrates.jax.distributions as tfd
import tensorflow_probability.substrates.jax.bijectors as tfb

In [5]:
def make_wishart(df, scale_tril):
    """Instantiates a WishartTriL distribution with Cholesky-space inputs.

    This factory function simplifies the creation of a Wishart distribution by 
    fixing 'validate_args' and 'input_output_cholesky' to ensure the model 
    operates entirely on lower-triangular matrices for numerical stability.

    Args:
        df: A tensor representing the degrees of freedom (must be > dim - 1).
        scale_tril: A tensor representing the lower-triangular Cholesky factor 
                    of the scale matrix (L, where Scale = LL').

    Returns:
        tfd.WishartTriL: A distribution object configured for Cholesky-space 
                         sampling and log-probability calculations.
    """
    return tfd.WishartTriL(
        df=df, 
        scale_tril=scale_tril, 
        input_output_cholesky=True, 
        validate_args=False
    )

def make_mvn_precision(loc, precision_factor):
    """Instantiates a Multivariate Normal distribution parameterized by precision.

    This factory handles the wrapping of a raw tensor into a LinearOperator, 
    allowing the MVN to be defined by its Precision (inverse covariance) 
    rather than its Covariance.

    The relationship between the precision matrix $P$ and the input is:
    $$P = L L^T$$
    where $L$ is the 'precision_factor'.

    Args:
        loc: A Tensor representing the mean vector(s).
        precision_factor: A Tensor representing the lower-triangular Cholesky 
            factor of the precision matrix.

    Returns:
        tfde.MultivariateNormalPrecisionFactorLinearOperator: An MVN 
            distribution object that uses the precision factor for 
            efficient computation.
    """
    return tfde.MultivariateNormalPrecisionFactorLinearOperator(
        loc=loc,
        precision_factor=tf.linalg.LinearOperatorLowerTriangular(precision_factor),
        validate_args=False
    )

In [ ]:
# --- 2. Vectorized Data Simulation ---

def simulate_data(seed=42, n_units=100, n_alts=5):
    """Simulates hierarchical multinomial logit data with unbalanced panel structure."""

    # Initialize random state and branch keys for independent sampling steps
    key = jax.random.PRNGKey(seed)
    k_beta, k_price, k_choice = jax.random.split(key, 3)
    
    ### Configuration: Create an unbalanced panel ###
    # Half of units have few observations, half have many (tests model robustness)
    # 4 Intercepts (one per alternative - one for the reference category) + 1 Price coefficient
    n_params = 5
    n_obs_small, n_obs_large = 5, 50

    # Compute total number of choice observations and unit mapping
    counts = jnp.array([n_obs_small] * (n_units // 2) + [n_obs_large] * (n_units // 2))
    total_obs = jnp.sum(counts)

    # Create unit index to map each observation row to its corresponding unit_id (0 to n_units-1)
    unit_idx = jnp.repeat(jnp.arange(n_units), counts)


    ### True population distribution parameters ###
    # Create mean vector and covariance matrix for the parameters
    # Add positive correlation between the 4th and 5th parameters (Price and Alt 5 Intercept)+
    true_mu = jnp.array([1.0, -1.0, 0.0, 0.0, -3.0])    
    true_Sigma = 3.0 * jnp.eye(n_params)
    true_Sigma = true_Sigma.at[3, 4].set(1.5).at[4, 3].set(1.5)

    ### Hierarchical Sampling: Draw unique beta vectors for every unit ###
    # Resulting shape: (n_units, n_params)
    beta_dist = tfd.MultivariateNormalFullCovariance(loc=true_mu, covariance_matrix=true_Sigma)
    betas = beta_dist.sample(seed=k_beta, sample_shape=(n_units,))
    

    ### Exogenous Features: Generate 'Price' for every observation/alternative ###
    # Resulting shape: (total_obs, n_alts)
    prices = jax.random.uniform(k_price, shape=(total_obs, n_alts), minval=-1.5, maxval=0.0)


    ### Design Matrix (X) Construction ###
    # Create alternative-specific intercepts (Identity matrix for first 4 params)
    identities = jnp.eye(n_alts, n_params - 1) 
    # Broadcast intercepts across all observations: (total_obs, 5, 4)
    X_fixed = jnp.broadcast_to(identities, (total_obs, n_alts, n_params - 1))
    # Append the price column as the 5th parameter: (total_obs, 5, 5)
    X = jnp.concatenate([X_fixed, prices[..., jnp.newaxis]], axis=-1)

    # 6. Choice Simulation
    # Expand unit-level betas to match the observation count: (total_obs, 5)
    obs_betas = betas[unit_idx] 
    
    # Calculate utility (logits): Multiply X[obs, alt, param] by beta[obs, param]
    # Summing over 'k' (params) gives shape (total_obs, n_alts)
    logits = jnp.einsum('ijk,ik->ij', X, obs_betas)
    
    # Sample the discrete choice (0-4) based on the Softmax of logits
    choices = tfd.Categorical(logits=logits).sample(seed=k_choice)

    return {
        "X": X,               # Predictors: (N_total, Alts, Params)
        "y": choices,         # Observed Choices: (N_total,)
        "betas": betas,       # Latent Truth: (N_units, Params)
        "unit_idx": unit_idx, # Unit Mapping: (N_total,)
        "true_mu": true_mu,
        "true_Sigma": true_Sigma
    }

In [7]:
# Genearete data
data = simulate_data()